# Namibia VACS 2019 — PUD exploration

Single public file **`NAMIBIA_VACS_2019_PUD.dta`** in **`data/raw/Namibia Stata/`**. Males and females share one dataset (**`SEX`**: 1 = male, 2 = female in this extract). Split-sample EAs (male vs female PSUs) per Data User Guide.

**Flow:** (1) Load → (2) §2 column list & quick EDA → (3) §3 samples & slot summaries → (4) §4 harmonized TSV (`variable_male` / `variable_female` columns list the **same** Stata names where the file is combined—see **notes**).

PDFs in folder: **`NAMIBIA_VACS_2019_DataUserGuide.pdf`**, codebooks, questionnaires.


In [ ]:
from pathlib import Path

from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyreadstat
import re

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

COUNTRY_DIR = ROOT / "data" / "raw" / "Namibia Stata"
PUD_PATH = COUNTRY_DIR / "NAMIBIA_VACS_2019_PUD.dta"

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")


## 1. Load data

`pyreadstat.read_dta` → `df`, `meta`.


In [ ]:
if not PUD_PATH.is_file():
    raise FileNotFoundError(f"Expected:\n  {PUD_PATH}")

df, meta = pyreadstat.read_dta(PUD_PATH)
print(f"File: {PUD_PATH}")
print(f"Rows × columns: {df.shape[0]:,} × {df.shape[1]:,}")

df = df.copy()
df["EA_HH"] = df["EA"].astype(float).astype(int).astype(str) + "_" + df["HH"].astype(float).astype(int).astype(str)

if "SEX" in df.columns:
    print("SEX (1=male, 2=female assumed):")
    display(df["SEX"].value_counts(dropna=False).sort_index())

print(f"VACS_ID unique / rows: {df['VACS_ID'].nunique():,} / {len(df):,}")
print(f"Duplicate VACS_ID rows: {int(df['VACS_ID'].duplicated().sum())}")
print(f"EA_HH unique / rows: {df['EA_HH'].nunique():,} / {len(df):,}")
print(f"Duplicate rows (all columns): {int(df.duplicated().sum())}")

df.head(4)


## 2. Column list & quick EDA

Stata labels, dtypes, missingness (first 40 + top missing).


In [ ]:
name_to_label = dict(meta.column_names_to_labels) if meta.column_names_to_labels else {}
var_table = pd.DataFrame({
    "column": df.columns,
    "stata_label": [name_to_label.get(c, "") or "" for c in df.columns],
    "dtype": df.dtypes.astype(str).values,
    "missing_n": df.isna().sum().values,
    "missing_pct": (100 * df.isna().mean()).round(2),
})
print(f"Variables: {len(df.columns):,}  |  Observations: {len(df):,}")
display(var_table.head(40))
display(var_table.sort_values("missing_pct", ascending=False).head(15).reset_index(drop=True))
df.info(max_cols=18)


## 3. Further EDA and exploration

### Raw row samples

IDs, geography, EA, strata, weights, interview date.


In [ ]:
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 72)

_core = [
    c
    for c in [
        "VACS_ID",
        "EA_HH",
        "EA",
        "HH",
        "REG",
        "REGION_NAME",
        "STRATA",
        "SEX",
        "SAMPLEWEIGHT",
        "HIVWEIGHT",
        "HDATE_F",
        "NTOT",
    ]
    if c in df.columns
]
sub = df[_core]
print("--- head(8) ---")
display(sub.head(8))
print("--- sample(6, random_state=2) ---")
display(sub.sample(6, random_state=2))


### Slot summaries

Width (**chars** vs integer-code digits), optional **style** line, **Male/Female text** heuristic (words/`_Female_` segments only—not single-letter **M**/**F** codes).

Duplicate **`VACS_ID`** example (if any) printed below summaries.


In [ ]:
L = meta.column_names_to_labels or {}

_WORD_SEX = re.compile(r"\b(?:male|females?|female)\b", re.IGNORECASE)
_EMBED_MF = re.compile(r"(?i)(?:^|_)(?:male|female)(?=_|$)")


def _abstract_digit_pattern(val: str) -> str:
    parts = []
    i = 0
    while i < len(val):
        ch = val[i]
        if ch.isdigit():
            j = i
            while j < len(val) and val[j].isdigit():
                j += 1
            parts.append("N" * (j - i))
            i = j
        elif ch.isalpha():
            j = i
            while j < len(val) and val[j].isalpha():
                j += 1
            parts.append("A")
            i = j
        else:
            parts.append(ch)
            i += 1
    return "".join(parts)


def _unified_style_pattern(st: pd.Series):
    st = st.dropna().astype(str)
    if len(st) == 0:
        return None
    abstracts = st.map(_abstract_digit_pattern)
    if abstracts.nunique(dropna=False) != 1:
        return None
    pat = abstracts.iloc[0]
    if not any(ch.isdigit() for ch in pat):
        return None
    if set(pat) <= {"N"}:
        return None
    ex = st.iloc[0]
    if len(pat) > 72:
        return f"{pat[:72]}… (e.g. {ex[:40]}{'…' if len(ex) > 40 else ''})"
    return f"{pat} (e.g. {ex})"


def _width_note(s: pd.Series) -> str:
    sn = s.dropna()
    if len(sn) == 0:
        return "n/a"
    if pd.api.types.is_numeric_dtype(s):
        whole = (sn == sn.astype(float).astype(int)).all()
        if whole:
            lens = sn.astype(int).astype(str).str.len()
            lo, hi = int(lens.min()), int(lens.max())
            return f"{lo}-{hi} digits (integer codes)" if lo != hi else f"{lo} digits (integer codes)"
        lens = sn.astype(str).str.len()
        lo, hi = int(lens.min()), int(lens.max())
        return f"{lo}-{hi} chars (numeric as string)" if lo != hi else f"{lo} chars (numeric as string)"
    st = sn.astype(str)
    lens = st.str.len()
    lo, hi = int(lens.min()), int(lens.max())
    w = f"{lo}-{hi} chars" if lo != hi else f"{lo} chars"
    if st.str.fullmatch(r"\d+").all():
        return f"{w} (string; all numeric characters)"
    return f"{w} (string)"


def _special_id_note(s: pd.Series) -> str:
    if pd.api.types.is_numeric_dtype(s):
        return "no M/F identifier (numeric)"
    st = s.dropna().astype(str)
    if len(st) == 0:
        return "n/a"
    if st.str.contains(_WORD_SEX, regex=True, na=False).any() or st.str.contains(_EMBED_MF, regex=True, na=False).any():
        return "Male/Female text (words or _Female_/_Male_ segments)"
    return "no male/female text (heuristic)"


def slot_summary(title: str, cols: list, note_extra: str = ""):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)
    miss = [c for c in cols if c not in df.columns]
    if miss:
        print("MISSING columns:", miss)
        return
    for c in cols:
        s = df[c]
        lbl = (L.get(c) or "")[:75]
        size_part = _width_note(s)
        style = _unified_style_pattern(s) if not pd.api.types.is_numeric_dtype(s) else None
        style_part = f"; style {style}" if style else ""
        id_part = _special_id_note(s)
        if pd.api.types.is_numeric_dtype(s):
            sn = s.dropna()
            extra = f"min/max={sn.min()}/{sn.max()}" if len(sn) else "min/max=n/a"
            info = f"{size_part}{style_part}; {id_part}; dtype={s.dtype}; {extra}"
        else:
            info = f"{size_part}{style_part}; {id_part}; dtype={s.dtype}"
        print(f"  {c} | {lbl}")
        print(f"    {info}; n_distinct={s.nunique(dropna=True)}; missing={s.isna().sum()}")
    if note_extra:
        print("  ", note_extra)


slot_summary("1. Respondent ID", ["VACS_ID"], "one duplicate ID pair in extract (inspect HDATE_F / row)")
slot_summary("2. Household / EA composite", ["EA", "HH", "EA_HH"], "")
slot_summary("3. Geo — region code + name", ["REG", "REGION_NAME"], "")
slot_summary("4. Geo / design — STRATA (region × urb/rural)", ["STRATA"], "")
slot_summary("5. Cluster / PSU — EA", ["EA"], "EA as enumeration area / PSU stage (see User Guide)")
slot_summary("6. Weights", ["SAMPLEWEIGHT", "HIVWEIGHT"], "")
slot_summary("7. Interview date", ["HDATE_F"], "")

print("\n--- Duplicate VACS_ID (if any) ---")
_d = df[df["VACS_ID"].duplicated(keep=False)].sort_values("VACS_ID")
if len(_d):
    _show = [c for c in ["VACS_ID", "SEX", "REG", "REGION_NAME", "EA", "HH", "HDATE_F", "SAMPLEWEIGHT"] if c in _d.columns]
    display(_d[_show])
else:
    print("none")


## 4. Harmonized codebook slots (Namibia 2019)

**One Excel row per slot.** **`variable_male`** and **`variable_female`** list the **same** Stata name(s) when there is a **single combined PUD**—use **`notes`** for **SEX** filtering (1 / 2) and duplicates.

**Source:** `data/raw/Namibia Stata/NAMIBIA_VACS_2019_PUD.dta`. Design: clustered sample; **split male/female EAs** (User Guide).

```
slot	variable_male	variable_female	type_and_width	notes
Respondent ID	VACS_ID	VACS_ID	str; ~27–35 chars; embeds region text + EA/HH-style tail	5190 distinct / 5191 rows—**one duplicate** `VACS_ID` (check `HDATE_F` empty vs filled); filter analysis with **SEX**
Household ID	EA + HH	EA + HH	int + int; composite **EA_HH** str for uniqueness	`EA` 1–274; `HH` 1–25 in extract; **EA_HH** ties ~1:1 to rows (one dup with duplicate ID); no separate `hh` id beyond **HH** within EA
Geo level 1	REG	REG	1–2 digits (integer); 14 codes	**Not 1:1** with **REGION_NAME** (same `REG` can pair with urban vs rural names—18 distinct REG×NAME pairs); use **REGION_NAME** or **STRATA** for full detail
Geo level 2	REGION_NAME	REGION_NAME	str; region + embedded Urban/Rural in name	16 distinct strings; urban/rural often in name (e.g. *Khomas - Urban*); **STRATA** parallels this split
Geo level 3	—	—	—	Optional: treat **STRATA** as geo/design cross-classification rather than a third admin boundary
Cluster / EA	EA	EA	1–3 digits (integer codes 1–274)	Enumeration area / PSU stage for clustering (confirm with User Guide + `svy` syntax)
Strata / selection	STRATA	STRATA	str; Region - Rural / Region - Urban pattern	27 levels; aligns with split-sample PSU design
Weight	SAMPLEWEIGHT	SAMPLEWEIGHT	float	HIV analysis weight **HIVWEIGHT** has missingness where not applicable (~946 miss)
Interview date	HDATE_F	HDATE_F	str; **YYYY-MM-DD** + `T00:00:00.` (Stata export; **20 chars** when filled)	Empty string possible; validate parsing to date; duplicate-ID pair differs on date field
```

**Excel:** For combined PUDs, duplicating the variable name in both male/female columns is intentional; analysts restrict with **SEX**.
